In [ ]:
#!/usr/bin/env python3
"""
Preprocessing pipeline to convert MSI and Visium h5ad files into PyG Data objects.
Designed for:
- MSI: TIC-normalized intensity features, coordinates in um (adata.obs['x_um','y_um'])
- Visium: library-normalized gene expression, reconstructed coords from tissue_positions CSV

Saves: processed_graphs/{modality}_{sample_id}.pt
"""

import os
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import scanpy as sc
import anndata
from scipy import sparse
from scipy.spatial import cKDTree
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import StandardScaler
import torch
from torch_geometric.data import Data
from tqdm import tqdm

# ----------------------------
# USER-CONFIG / DEFAULT PATHS
# ----------------------------
MSI_INPUT_FOLDER = "/home/ajarrah/PhD_Thesis/chapter_2/h5ad_data_processed_4lockmasses_filtered_halfbrain/"
COMMON_MZS_CSV = "/home/ajarrah/PhD_Thesis/chapter_2/csv_data/common_mz_clusters_improved.csv"
MSI_SAMPLE_IDS = [
    "yc_1", "yc_2", "yc_3", "yc_4",
    "yad_1", "yad_2", "yad_3", "yad_4",
    "ac_1", "ac_2", "ac_3", "ac_4",
    "aad_1", "aad_2", "aad_3", "aad_4"
]

RNA_INPUT_FOLDER = "/home/ajarrah/PhD_Thesis/chapter_3/h5ad_data/"
RNA_TISSUE_POSITIONS_CSV = "/home/ajarrah/PhD_Thesis/chapter_4/tissue_positions/tissue_positions.csv"
RNA_SAMPLE_FILES = [
    "A1_Young_Control_Mouse_Brain_202502.h5ad",
    "B1_Young_Control_Mouse_Brain_202502.h5ad",
    "C1_Young_Control_Mouse_Brain_202502.h5ad",
    "D1_Young_Control_Mouse_Brain_202502.h5ad",
    "A1_Young_AD_Mouse_Brain_202502.h5ad",
    "B1_Young_AD_Mouse_Brain_202502.h5ad",
    "C1_Young_AD_Mouse_Brain_202502.h5ad",
    "D1_Young_AD_Mouse_Brain_202502.h5ad",
    "A1_Aged_Control_Mouse_Brain_202502.h5ad",
    "B1_Aged_Control_Mouse_Brain_202502.h5ad",
    "C1_Aged_Control_Mouse_Brain_202502.h5ad",
    "D1_Aged_Control_Mouse_Brain_202502.h5ad",
    "A1_Aged_AD_Mouse_Brain_202502.h5ad",
    "B1_Aged_AD_Mouse_Brain_202502.h5ad",
    "C1_Aged_AD_Mouse_Brain_202502.h5ad",
    "D1_Aged_AD_Mouse_Brain_202502.h5ad"
]
RNA_SAMPLE_IDS = [
    "YC_1","YC_2","YC_3","YC_4",
    "YAD_1","YAD_2","YAD_3","YAD_4",
    "AC_1","AC_2","AC_3","AC_4",
    "AAD_1","AAD_2","AAD_3","AAD_4"
]

OUTPUT_DIR = "processed_graphs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ----------------------------
# PARAMETERS (tuneable)
# ----------------------------
RADIUS_UM = 110.0           # spatial radius in microns for graph edges (tuneable)
REDUCE_DIM = True           # whether to perform TruncatedSVD/PCA on features
N_COMPONENTS = 128          # target feature dimension after reduction
LOG_TRANSFORM = True        # apply log1p to normalized intensities
VISIUM_HVG_N = 2000         # number of HVGs to keep for Visium (if available)
MSI_TOP_K_MZ = None         # if you want to select top-k m/z per sample (None = use COMMON_MZS_CSV or all)
USE_COMMON_MZS = True       # use COMMON_MZS_CSV to select m/z columns for MSI if available

# ----------------------------
# Helpers
# ----------------------------
def ensure_numpy_matrix(X):
    """Return dense numpy array from various possible storages (ndarray, sparse, DataFrame)."""
    if sparse.issparse(X):
        return X.toarray()
    if isinstance(X, np.ndarray):
        return X
    if hasattr(X, "values"):
        return X.values
    return np.asarray(X)

def tic_normalize(intensity_matrix, tic_values):
    """Divide each observation (row) by its TIC; intensity_matrix shape: n_obs x n_features"""
    # Prevent division by zero
    tic = np.array(tic_values, dtype=float)
    tic[tic == 0] = 1.0
    X_norm = intensity_matrix / tic[:, None]
    return X_norm

def build_radius_edges(coords, radius):
    """
    coords: (N,2) numpy array in microns
    returns edge_index tensor shaped [2, E] (bidirectional)
    """
    tree = cKDTree(coords)
    neighbors = tree.query_ball_point(coords, r=radius)
    # Build edge list
    row_idx = []
    col_idx = []
    for i, nbrs in enumerate(neighbors):
        for j in nbrs:
            if i == j:
                continue
            row_idx.append(i)
            col_idx.append(j)
    if len(row_idx) == 0:
        # fallback: fully connect small graphs
        N = coords.shape[0]
        row_idx = []
        col_idx = []
        for i in range(N):
            for j in range(N):
                if i != j:
                    row_idx.append(i)
                    col_idx.append(j)
    edge_index = torch.tensor([row_idx, col_idx], dtype=torch.long)
    return edge_index

def reduce_features(X, n_components=N_COMPONENTS):
    """
    X: numpy array (n_nodes x n_features)
    Uses TruncatedSVD for dense/sparse friendliness -> returns numpy (n_nodes x n_components)
    """
    # Standardize first (column-wise)
    scaler = StandardScaler(with_mean=False)  # with_mean=False to keep sparse compatibility
    Xs = scaler.fit_transform(X)
    svd = TruncatedSVD(n_components=min(n_components, Xs.shape[1]-1 or 1), random_state=0)
    X_reduced = svd.fit_transform(Xs)
    return X_reduced.astype(np.float32)

# ----------------------------
# MSI processing
# ----------------------------
def process_msi_h5ad(path_h5ad, sample_id, common_mzs_csv=None,
                     use_common_mzs=True, top_k_mz=None,
                     radius_um=RADIUS_UM,
                     log_transform=LOG_TRANSFORM,
                     reduce_dim=REDUCE_DIM):
    """
    Input: path to MSI h5ad (AnnData). Assumes:
      - adata.X or adata.layers[...] contains intensities (observations x m/z)
      - adata.obs contains 'x_um', 'y_um', 'Processed_TIC' columns
      - adata.var contains m/z values in 'mzs' or index
    """
    print(f"[MSI] Loading {path_h5ad}")
    adata = sc.read_h5ad(path_h5ad)
    # find intensity matrix:
    if adata.X is None:
        raise ValueError("adata.X is None for MSI file: " + str(path_h5ad))
    X_raw = ensure_numpy_matrix(adata.X)  # n_obs x n_var
    # coordinates
    if "x_um" not in adata.obs.columns or "y_um" not in adata.obs.columns:
        raise KeyError("adata.obs must contain 'x_um' and 'y_um' columns for MSI sample " + sample_id)
    coords = np.vstack([adata.obs["x_um"].values, adata.obs["y_um"].values]).T.astype(float)
    # TIC normalization
    if "Processed_TIC" in adata.obs.columns:
        tic = adata.obs["Processed_TIC"].values
    elif "TIC" in adata.obs.columns:
        tic = adata.obs["TIC"].values
    else:
        # try sum across spectrum
        tic = X_raw.sum(axis=1)
    X_norm = tic_normalize(X_raw.astype(float), tic)
    # choose features (common m/z or top-k)
    if use_common_mzs and (common_mzs_csv is not None) and os.path.exists(common_mzs_csv):
        df_mz = pd.read_csv(common_mzs_csv)
        # Expect the csv to have a column 'mz' or 'm/z' or 'center_mz'
        mz_col = None
        for candidate in ["mz", "m/z", "center_mz", "mean_mz", "mz_center"]:
            if candidate in df_mz.columns:
                mz_col = candidate
                break
        if mz_col is None:
            # if no header, assume single column
            df_mz.columns = ["mz"]
            mz_col = "mz"
        common_mz_list = df_mz[mz_col].values.astype(float)
        # Match adata.var (m/z) to common_mz_list by nearest neighbor
        var_mzs = None
        if "mzs" in adata.var.columns:
            var_mzs = adata.var["mzs"].values.astype(float)
        else:
            try:
                var_mzs = adata.var.index.astype(float)
            except Exception:
                var_mzs = np.arange(adata.shape[1])
        # For each common mz, find nearest var index
        idxs = []
        for mz in common_mz_list:
            idx = np.argmin(np.abs(var_mzs - mz))
            idxs.append(idx)
        idxs = np.unique(idxs)
        X_sel = X_norm[:, idxs]
    elif top_k_mz is not None:
        # choose top-k m/z by global mean intensity
        mean_by_mz = X_norm.mean(axis=0)
        idxs = np.argsort(mean_by_mz)[::-1][:top_k_mz]
        X_sel = X_norm[:, idxs]
    else:
        X_sel = X_norm

    # log transform and nan handling
    if log_transform:
        X_sel = np.log1p(X_sel)
    X_sel = np.nan_to_num(X_sel, nan=0.0, posinf=0.0, neginf=0.0).astype(np.float32)

    # feature reduction
    if reduce_dim and X_sel.shape[1] > N_COMPONENTS:
        X_proc = reduce_features(X_sel, n_components=N_COMPONENTS)
    else:
        X_proc = X_sel.astype(np.float32)

    # build edges with radius graph
    edge_index = build_radius_edges(coords, radius_um)

    # create PyG Data
    data = Data(
        x=torch.tensor(X_proc, dtype=torch.float),
        pos=torch.tensor(coords, dtype=torch.float),
        edge_index=edge_index
    )
    # metadata
    data.sample_id = sample_id
    data.modality = "MSI"
    data.n_nodes = X_proc.shape[0]
    data.n_features = X_proc.shape[1]
    return data

# ----------------------------
# Visium processing
# ----------------------------
def reconstruct_visium_coords_from_tissue_positions(tissue_positions_csv, spot_spacing_um=100.0):
    """
    Reconstruct cartesian coords (x_um,y_um) from array_row/array_col in tissue_positions.csv
    using hex layout mapping:
      y = row * (sqrt(3)/2 * spot_spacing)
      x = col * (spot_spacing / 2)
    Assumes CSV has columns: ['barcode','array_row','array_col'] or similar.
    """
    df = pd.read_csv(tissue_positions_csv, usecols=["barcode", "array_row", "array_col"])
    rows = df["array_row"].values.astype(float)
    cols = df["array_col"].values.astype(float)
    # hex spacing mapping (this matches your earlier code)
    y = rows * (np.sqrt(3)/2.0 * spot_spacing_um)
    x = cols * (spot_spacing_um / 2.0)
    coords_um = np.vstack([x, y]).T
    df["x_um"] = coords_um[:, 0]
    df["y_um"] = coords_um[:, 1]
    return df.set_index("barcode")[["x_um","y_um"]]

def process_visium_h5ad(path_h5ad, sample_id, tissue_positions_df,
                        radius_um=RADIUS_UM,
                        log_transform=LOG_TRANSFORM,
                        reduce_dim=REDUCE_DIM,
                        hvg_n=VISIUM_HVG_N):
    """
    Input: Visium h5ad with obs index barcodes that match tissue_positions_df
    Steps:
      - normalize_total (library size) -> adata.X
      - optionally select highly-variable genes (HVGs)
      - optionally log1p
      - build radius graph using reconstructed x_um,y_um
    """
    print(f"[Visium] Loading {path_h5ad}")
    adata = sc.read_h5ad(path_h5ad)
    # Normalize library size (if not already)
    sc.pp.normalize_total(adata, target_sum=1e4)  # counts per 10k
    if log_transform:
        sc.pp.log1p(adata)

    # find coordinates for each spot using barcode index lookup
    # tissue_positions_df indexed by barcode -> x_um,y_um
    if "x_um" in adata.obs.columns and "y_um" in adata.obs.columns:
        coords = np.vstack([adata.obs["x_um"].values, adata.obs["y_um"].values]).T.astype(float)
    else:
        # try to match by barcode
        barcodes = adata.obs_names
        coords = []
        missing = 0
        for bc in barcodes:
            if bc in tissue_positions_df.index:
                x, y = tissue_positions_df.loc[bc, ["x_um","y_um"]].values
                coords.append((x,y))
            else:
                coords.append((np.nan, np.nan))
                missing += 1
        coords = np.array(coords).astype(float)
        if missing > 0:
            print(f"[Visium] Warning: {missing} barcodes missing from tissue_positions; their coordinates are NaN.")
        # if NaNs exist, try to fall back to adata.obs['array_row','array_col'] if present
        if np.isnan(coords).any():
            if "array_row" in adata.obs.columns and "array_col" in adata.obs.columns:
                r = adata.obs["array_row"].values.astype(float)
                c = adata.obs["array_col"].values.astype(float)
                y = r * (np.sqrt(3)/2.0 * 100.0)
                x = c * (100.0 / 2.0)
                coords = np.vstack([x,y]).T.astype(float)

    # features matrix
    X_raw = ensure_numpy_matrix(adata.X)
    # optional HVG selection
    if hvg_n is not None and adata.shape[1] > hvg_n:
        try:
            sc.pp.highly_variable_genes(adata, flavor="seurat_v3", n_top_genes=hvg_n, subset=True)
            X_sel = ensure_numpy_matrix(adata.X)
        except Exception:
            # fallback: variance-based selection
            var = X_raw.var(axis=0)
            idx = np.argsort(var)[::-1][:hvg_n]
            X_sel = X_raw[:, idx]
    else:
        X_sel = X_raw

    X_sel = np.nan_to_num(X_sel, nan=0.0, posinf=0.0, neginf=0.0).astype(np.float32)

    # optional reduction
    if reduce_dim and X_sel.shape[1] > N_COMPONENTS:
        X_proc = reduce_features(X_sel, n_components=N_COMPONENTS)
    else:
        X_proc = X_sel.astype(np.float32)

    # build graph edges
    # remove nodes with NaN coords (if any)
    valid_mask = ~np.isnan(coords).any(axis=1)
    coords_valid = coords[valid_mask]
    X_valid = X_proc[valid_mask]

    edge_index = build_radius_edges(coords_valid, radius_um)

    data = Data(
        x=torch.tensor(X_valid, dtype=torch.float),
        pos=torch.tensor(coords_valid, dtype=torch.float),
        edge_index=edge_index
    )
    data.sample_id = sample_id
    data.modality = "Visium"
    data.n_nodes = X_valid.shape[0]
    data.n_features = X_valid.shape[1]
    return data

# ----------------------------
# Main orchestration
# ----------------------------
def main():
    # Load common mz CSV if requested
    common_mzs_csv = COMMON_MZS_CSV if (USE_COMMON_MZS and os.path.exists(COMMON_MZS_CSV)) else None
    if common_mzs_csv:
        print("[Main] Using common m/z CSV:", common_mzs_csv)
    else:
        print("[Main] No common m/z CSV used (either disabled or missing).")

    # Process MSI samples
    print("[Main] Processing MSI samples...")
    for sid in tqdm(MSI_SAMPLE_IDS, desc="MSI samples"):
        h5ad_path = os.path.join(MSI_INPUT_FOLDER, f"{sid}.h5ad")
        if not os.path.exists(h5ad_path):
            # try uppercase or alternate naming
            alt = os.path.join(MSI_INPUT_FOLDER, f"{sid.lower()}.h5ad")
            if os.path.exists(alt):
                h5ad_path = alt
            else:
                print(f"[MSI] WARNING: {h5ad_path} not found. Skipping sample {sid}.")
                continue
        data = process_msi_h5ad(h5ad_path, sample_id=sid, common_mzs_csv=common_mzs_csv,
                                use_common_mzs=USE_COMMON_MZS, top_k_mz=MSI_TOP_K_MZ,
                                radius_um=RADIUS_UM,
                                log_transform=LOG_TRANSFORM,
                                reduce_dim=REDUCE_DIM)
        out_path = os.path.join(OUTPUT_DIR, f"msi_{sid}.pt")
        torch.save(data, out_path)
        print(f"[MSI] Saved graph: {out_path} (nodes={data.n_nodes}, feat_dim={data.n_features})")

    # Prepare tissue positions mapping for Visium
    print("[Main] Loading Visium tissue positions...")
    if os.path.exists(RNA_TISSUE_POSITIONS_CSV):
        tissue_pos_df = reconstruct_visium_coords_from_tissue_positions(RNA_TISSUE_POSITIONS_CSV, spot_spacing_um=100.0)
        print("[Main] Tissue positions loaded, entries:", len(tissue_pos_df))
    else:
        print("[Main] WARNING: tissue_positions CSV not found at:", RNA_TISSUE_POSITIONS_CSV)
        tissue_pos_df = pd.DataFrame()

    # Process Visium samples
    print("[Main] Processing Visium samples...")
    for idx, fname in enumerate(tqdm(RNA_SAMPLE_FILES, desc="Visium samples")):
        sample_id = RNA_SAMPLE_IDS[idx] if idx < len(RNA_SAMPLE_IDS) else f"RNA_{idx}"
        h5ad_path = os.path.join(RNA_INPUT_FOLDER, fname)
        if not os.path.exists(h5ad_path):
            print(f"[Visium] WARNING: {h5ad_path} not found. Skipping.")
            continue
        data = process_visium_h5ad(h5ad_path, sample_id=sample_id, tissue_positions_df=tissue_pos_df,
                                   radius_um=RADIUS_UM,
                                   log_transform=LOG_TRANSFORM,
                                   reduce_dim=REDUCE_DIM,
                                   hvg_n=VISIUM_HVG_N)
        out_path = os.path.join(OUTPUT_DIR, f"visium_{sample_id}.pt")
        torch.save(data, out_path)
        print(f"[Visium] Saved graph: {out_path} (nodes={data.n_nodes}, feat_dim={data.n_features})")

    print("[Main] Preprocessing finished. Processed graphs saved in:", OUTPUT_DIR)

if __name__ == "__main__":
    main()
